# Module 04 — RLHF with PPO

**Goal**: Use the reward model from Module 03 to optimise a policy with Proximal Policy Optimisation (PPO).

**What you will build**:
1. Load an SFT model as the *policy*
2. Load a reward model as the *scorer*
3. Run a PPO training loop: generate → score → update
4. Watch the average reward climb over time

In [ ]:
# Cell 1 — Install
!pip install -q transformers datasets trl peft accelerate torch

## 1. RLHF in one diagram

```
┌─────────────────────────────────────────────────────────┐
│  RLHF Loop (one step)                                   │
│                                                         │
│  [Prompt]                                               │
│     │                                                   │
│     ▼                                                   │
│  Policy LM  ──► generates response                      │
│     │                          │                        │
│     │                          ▼                        │
│     │              Reward Model ──► scalar r            │
│     │                          │                        │
│     ▼                          │                        │
│  PPO update ◄──────────────────┘                       │
│  (maximise r, stay close to SFT reference)              │
└─────────────────────────────────────────────────────────┘
```

**KL penalty**: PPO adds `β * KL(policy || reference)` to prevent the model
from "reward hacking" — exploiting weaknesses in the reward model rather than
genuinely improving.

**PPO clip**: The policy update is clipped so the new policy doesn't deviate
too far from the old one in a single step, ensuring stable training.

In [ ]:
# Cell 2 — Imports
import torch
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer
from datasets import Dataset
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead
import numpy as np

POLICY_ID  = "HuggingFaceTB/SmolLM2-135M-Instruct"  # SFT model as starting point
REWARD_ID  = "HuggingFaceTB/SmolLM2-135M"           # base for reward head
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE      = torch.bfloat16
PPO_STEPS  = 40   # each step = generate + score + update
print(f"Device: {DEVICE}")

## 2. Load policy model

PPO needs a model with a **value head** — an extra scalar output that estimates
how good the current state is (the critic). TRL provides `AutoModelForCausalLMWithValueHead`.

In [ ]:
# Cell 3 — Policy model with value head
tokenizer = AutoTokenizer.from_pretrained(POLICY_ID)
tokenizer.pad_token = tokenizer.eos_token

policy = AutoModelForCausalLMWithValueHead.from_pretrained(
    POLICY_ID,
    torch_dtype=DTYPE,
    device_map=DEVICE,
)
print("Policy model loaded (LM head + value head)")

## 3. Load reward model

In [ ]:
# Cell 4 — Reward model (from Module 03 or a pretrained one)
# For this demo we use the base model + classification head as a stand-in.
# In production: load your trained reward model from Module 03.
reward_model = AutoModelForSequenceClassification.from_pretrained(
    REWARD_ID,
    num_labels=1,
    torch_dtype=DTYPE,
    device_map=DEVICE,
)
reward_model.config.pad_token_id = tokenizer.eos_token_id
reward_model.eval()
print("Reward model loaded")

## 4. Build a prompt dataset

PPO needs a set of prompts to generate responses from during training.

In [ ]:
# Cell 5 — Prompt dataset
PROMPTS = [
    "Explain what a language model is.",
    "What is the difference between supervised and unsupervised learning?",
    "Describe the transformer architecture briefly.",
    "What is cross-entropy loss?",
    "Explain what fine-tuning means in deep learning.",
    "What is the purpose of an activation function?",
    "Describe the attention mechanism.",
    "What is the difference between training and inference?",
    "Explain regularisation in machine learning.",
    "What is a convolutional neural network used for?",
] * 4   # 40 prompts

prompt_ds = Dataset.from_dict({"query": PROMPTS})
print(f"Prompt dataset: {len(prompt_ds)} prompts")

## 5. PPO training loop

In [ ]:
# Cell 6 — PPOConfig
ppo_config = PPOConfig(
    learning_rate=1e-5,
    batch_size=4,
    mini_batch_size=2,
    gradient_accumulation_steps=1,
    optimize_cuda_cache=True,
    kl_penalty="kl",
    init_kl_coef=0.2,       # β — how much to penalise KL divergence
    target_kl=6.0,          # adaptive KL target
    cliprange=0.2,           # PPO clip ε
    cliprange_value=0.2,
    vf_coef=0.1,             # value function loss weight
    log_with=None,
    report_to="none",
)

In [ ]:
# Cell 7 — PPO training loop
ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=policy,
    ref_model=None,          # TRL creates a frozen copy automatically
    tokenizer=tokenizer,
    dataset=prompt_ds,
    data_collator=lambda x: x,
)

reward_history = []

def get_reward_score(texts):
    """Score a list of texts with the reward model."""
    scores = []
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt",
                           truncation=True, max_length=256,
                           padding=True).to(reward_model.device)
        with torch.no_grad():
            score = reward_model(**inputs).logits[0].item()
        scores.append(torch.tensor(score, dtype=torch.float32))
    return scores

print(f"Running {PPO_STEPS} PPO steps...")
for step, batch in enumerate(ppo_trainer.dataloader):
    if step >= PPO_STEPS:
        break

    # Step 1 — generate responses from the policy
    query_tensors = [
        tokenizer(q, return_tensors="pt").input_ids.squeeze(0).to(policy.device)
        for q in batch["query"]
    ]
    response_tensors = ppo_trainer.generate(
        query_tensors,
        max_new_tokens=80,
        temperature=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    # Step 2 — decode and score
    responses = [tokenizer.decode(r, skip_special_tokens=True)
                 for r in response_tensors]
    full_texts = [q + " " + r for q, r in zip(batch["query"], responses)]
    rewards = get_reward_score(full_texts)

    # Step 3 — PPO update
    stats = ppo_trainer.step(query_tensors, response_tensors, rewards)

    mean_reward = torch.stack(rewards).mean().item()
    reward_history.append(mean_reward)

    if step % 10 == 0:
        print(f"Step {step:3d}  |  mean reward: {mean_reward:.4f}  |  kl: {stats.get('objective/kl', 0):.4f}")

print("\nPPO training complete!")

In [ ]:
# Cell 8 — Plot reward over training
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4))
plt.plot(reward_history, color="#e76f51", linewidth=2, label="Mean reward per step")
# Smooth
if len(reward_history) > 5:
    smooth = np.convolve(reward_history, np.ones(5)/5, mode="valid")
    plt.plot(range(4, len(reward_history)), smooth,
             color="#264653", linewidth=2.5, linestyle="--", label="Smoothed (window=5)")
plt.xlabel("PPO Step")
plt.ylabel("Mean Reward")
plt.title("PPO Training — Reward Over Time")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/ppo_reward.png", dpi=120)
plt.show()
print(f"Reward: {reward_history[0]:.4f} → {reward_history[-1]:.4f}")

## ✅ Summary

| Concept | Detail |
|---------|--------|
| **Policy** | SFT model + value head; generates responses |
| **Reference** | Frozen copy of the policy at start; provides KL baseline |
| **Reward** | Scalar from reward model — what we're maximising |
| **KL penalty** | Prevents reward hacking by keeping policy near the reference |
| **PPO clip** | Prevents the policy update from being too large in one step |

**Next**: Module 05 implements DPO — which achieves similar results *without* a reward model or RL loop, by directly optimising on preference pairs.

> 💡 PPO is powerful but complex and computationally expensive. For most fine-tuning tasks DPO (Module 05) is simpler and competitive.